# Symbiote: corrected scorer training

Historical completed workflow. Existing final results need no retraining; read README.md before an intentional reproduction.
Attach the private Dataset made from `symbiote_kaggle_start_20261021.zip`.
Select a GPU and enable Internet. Run cells in order.

This notebook trains on saved, regraded labels. It makes **no labeling or
target-model API calls**. DistilBERT weights are downloaded on Kaggle.
The last cell exports results and weights for download. If a command fails,
the notebook attempts to export partial outputs before raising the error.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
import torch

assert torch.cuda.is_available(), "No CUDA GPU. Enable a Kaggle GPU before continuing."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"
INPUT_ROOT = Path("/kaggle/input")
ROOT = Path("/kaggle/working/symbiote")
ROOT.mkdir(exist_ok=True)

archives = list(INPUT_ROOT.rglob("symbiote_kaggle_start_20261021.zip"))
if len(archives) > 1:
    raise RuntimeError("Attach only one copy of the Symbiote starter Dataset")
if archives:
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            (ROOT / member.filename).resolve().relative_to(ROOT.resolve())
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError("Unexpected symlink in starter bundle")
        archive.extractall(ROOT)
else:
    manifests = []
    for path in INPUT_ROOT.rglob("BOOTSTRAP_MANIFEST.json"):
        data = json.loads(path.read_text())
        if data.get("kind") == "symbiote-kaggle-stage-a3":
            manifests.append(path)
    if len(manifests) != 1:
        raise RuntimeError("Attach the private Symbiote starter Dataset before running")
    origin = manifests[0].parent
    data = json.loads(manifests[0].read_text())
    for relative in list(data["files"]) + ["BOOTSTRAP_MANIFEST.json"]:
        source = (origin / relative).resolve()
        source.relative_to(origin.resolve())
        target = (ROOT / relative).resolve()
        target.relative_to(ROOT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)

manifest = json.loads((ROOT / "BOOTSTRAP_MANIFEST.json").read_text())
assert manifest["kind"] == "symbiote-kaggle-stage-a3"
for relative, expected in manifest["files"].items():
    path = (ROOT / relative).resolve()
    path.relative_to(ROOT.resolve())
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected, f"Checksum mismatch: {relative}"
os.chdir(ROOT)
print("Verified", len(manifest["files"]), "files in", ROOT)


In [ ]:
# Keep Kaggle's working CUDA/PyTorch stack. Install only missing training tools.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "pandas", "numpy", "scikit-learn", "matplotlib", "requests", "python-dotenv"], check=True)
assert torch.cuda.is_available(), "CUDA became unavailable; stop and inspect the environment"
PREPARED = "data/processed/team_v2_20261021"
MODEL = "models/span_scorer_v2_20261021"
RUN = Path("outputs/final_eval")
RUN.mkdir(parents=True, exist_ok=True)

def export_results():
    destination = Path("/kaggle/working/stage_a3_results.zip")
    folders = [Path(MODEL), RUN, Path("outputs/offline_preparation_20261021"), Path(PREPARED)]
    files = sorted({p for folder in folders if folder.exists() for p in folder.rglob("*") if p.is_file()})
    hashes = {}
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in files:
            if path.name == ".env" or path.suffix == ".env":
                continue
            relative = path.relative_to(ROOT) if path.is_absolute() else path
            archive.write(path, relative.as_posix())
            with path.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            hashes[relative.as_posix()] = digest
        archive.write(ROOT / "BOOTSTRAP_MANIFEST.json", "BOOTSTRAP_MANIFEST.json")
        archive.writestr("RESULTS_MANIFEST.json", json.dumps({"files": hashes}, indent=2))
    print("Download:", destination, "bytes:", destination.stat().st_size)
    return destination

def run_module(module, arguments, log_name):
    command = [sys.executable, "-u", "-m", module, *arguments]
    print("Running:", " ".join(command), flush=True)
    with (RUN / log_name).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            export_results()
            raise
    if status:
        export_results()
        raise RuntimeError(f"Command failed ({status}); inspect {log_name} and download partial results")

run_module("final_eval.handoff", ["environment", "--output", str(RUN / "environment.json")], "environment.log")


## Validate the real Kaggle environment
The dry-run must show 13,966 spans, 912 questions, 778 base families and
fractional targets. This step loads no model weights. The following cell
performs the five-fold GPU training; leave its settings fixed for this run.

In [ ]:
TRAIN = ["--consensus-file", f"{PREPARED}/consensus_dataset.csv",
         "--output-dir", MODEL, "--folds", "5", "--batch-size", "8", "--epochs", "4"]
run_module("final_eval.scorer.train_scorer", [*TRAIN, "--dry-run"], "scorer_dry_run.log")


In [ ]:
# Only completed folds can be resumed. The trainer verifies identical metadata.
resume = ["--resume"] if Path(MODEL).exists() else []
run_module("final_eval.scorer.train_scorer", [*TRAIN, *resume], "scorer_training.log")


In [ ]:
run_module("final_eval.analysis.feature_baseline", [
    "--consensus-file", f"{PREPARED}/consensus_dataset.csv", "--model-dir", MODEL,
    "--output-dir", str(RUN / "feature_baseline")], "feature_baseline.log")
run_module("final_eval.preparation.cohort", [
    "--problems", f"{PREPARED}/problems.csv", "--split", f"{MODEL}/split.json",
    "--output", str(RUN / "pilot/cohort.csv"), "--max-questions", "16"], "pilot_cohort.log")
export_results()


Download `/kaggle/working/stage_a3_results.zip`, including checkpoint weights,
before ending this GPU session. Return that ZIP to the local project/chat.
This completes training, the feature comparison and pilot cohort selection;
it does not run compressors or downstream target evaluation. Those stages
will use these exact checkpoints and their held-out question assignments.

To export a partial run manually after an interruption, run `export_results()`
in a new cell while this working directory still exists. Keep trained weights
in saved Kaggle outputs as well if your account offers that option.